# Student Performance Prediction Using ANN

This notebook implements an Artificial Neural Network (ANN) to predict student grades based on various academic and lifestyle factors.

## 1. Importing Essential Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## 2. Load Dataset

In [2]:
try:
    df = pd.read_csv('Students Performance Dataset.csv')
    print("Dataset loaded successfully.")
    print(f"Dataset shape: {df.shape}")
    display(df.head())
    print("\nDataset Info:")
    print(df.info())
except FileNotFoundError:
    print("Error: Dataset file 'Students Performance Dataset.csv' not found.")
    print("Please ensure the dataset is in the same directory as this notebook.")

Dataset loaded successfully.
Dataset shape: (5000, 23)


,Student_ID,First_Name,Last_Name,Email,Gender,Age,Department,Attendance (%),Midterm_Score,Final_Score,...,Projects_Score,Total_Score,Grade,Study_Hours_per_Week,Extracurricular_Activities,Internet_Access_at_Home,Parent_Education_Level,Family_Income_Level,Stress_Level (1-10),Sleep_Hours_per_Night
0,S1000,Omar,Williams,student0@university.com,Female,22,Mathematics,97.36,40.61,59.61,...,62.84,59.8865,F,10.3,Yes,No,Master's,Medium,1,5.9
1,S1001,Maria,Brown,student1@university.com,Male,18,Business,97.71,57.27,74.00,...,98.23,81.9170,B,27.1,No,No,High School,Low,4,4.3
2,S1002,Ahmed,Jones,student2@university.com,Male,24,Engineering,99.52,41.84,63.85,...,91.22,67.7170,D,12.4,Yes,No,High School,Low,9,6.1
3,S1003,Omar,Williams,student3@university.com,Female,24,Engineering,90.38,45.65,44.44,...,55.48,51.6535,F,25.5,No,Yes,High School,Low,8,4.9
4,S1004,John,Smith,student4@university.com,Female,23,CS,59.41,53.13,61.77,...,87.43,71.4030,C,13.3,Yes,No,Master's,Medium,6,4.5



Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Student_ID                  5000 non-null   str    
 1   First_Name                  5000 non-null   str    
 2   Last_Name                   5000 non-null   str    
 3   Email                       5000 non-null   str    
 4   Gender                      5000 non-null   str    
 5   Age                         5000 non-null   int64  
 6   Department                  5000 non-null   str    
 7   Attendance (%)              5000 non-null   float64
 8   Midterm_Score               5000 non-null   float64
 9   Final_Score                 5000 non-null   float64
 10  Assignments_Avg             5000 non-null   float64
 11  Quizzes_Avg                 5000 non-null   float64
 12  Participation_Score         5000 non-null   float64
 13  Projects_Score              5

## 3. Data Preprocessing

In [3]:
# Drop irrelevant columns (identifiers and personal info)
drop_cols = ['Student_ID', 'First_Name', 'Last_Name', 'Email']
df = df.drop(columns=drop_cols)

print("Columns after dropping identifiers:")
print(df.columns.tolist())

Columns after dropping identifiers:
['Gender', 'Age', 'Department', 'Attendance (%)', 'Midterm_Score', 'Final_Score', 'Assignments_Avg', 'Quizzes_Avg', 'Participation_Score', 'Projects_Score', 'Total_Score', 'Grade', 'Study_Hours_per_Week', 'Extracurricular_Activities', 'Internet_Access_at_Home', 'Parent_Education_Level', 'Family_Income_Level', 'Stress_Level (1-10)', 'Sleep_Hours_per_Night']


### 3.1 Handle Missing Values

In [4]:
# Check for missing values
print("Missing values before imputation:")
print(df.isnull().sum())

# Impute categorical columns with mode
categorical_cols_all = df.select_dtypes(include=['object']).columns
for col in categorical_cols_all:
    if df[col].isnull().sum() > 0:
        print(f"Imputing missing values for '{col}' with mode.")
        df[col] = df[col].fillna(df[col].mode()[0])

# Impute numerical columns with mean
numerical_cols_all = df.select_dtypes(include=['number']).columns
for col in numerical_cols_all:
    if df[col].isnull().sum() > 0:
        print(f"Imputing missing values for '{col}' with mean.")
        df[col] = df[col].fillna(df[col].mean())

print("\nMissing values after imputation:")
print(df.isnull().sum().sum())

Missing values before imputation:
Gender                           0
Age                              0
Department                       0
Attendance (%)                   0
Midterm_Score                    0
Final_Score                      0
Assignments_Avg                  0
Quizzes_Avg                      0
Participation_Score              0
Projects_Score                   0
Total_Score                      0
Grade                            0
Study_Hours_per_Week             0
Extracurricular_Activities       0
Internet_Access_at_Home          0
Parent_Education_Level        1025
Family_Income_Level              0
Stress_Level (1-10)              0
Sleep_Hours_per_Night            0
dtype: int64
Imputing missing values for 'Parent_Education_Level' with mode.

Missing values after imputation:
0


C:\Users\Asus\AppData\Local\Temp\ipykernel_22916\4040277163.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_all = df.select_dtypes(include=['object']).columns


### 3.2 Separate Features and Target

In [ ]:
# Separate Features (X) and Target (y)
# Drop 'Total_Score' and 'Final_Score' to avoid data leakage
# (Grade is likely derived from these scores)
X = df.drop(columns=['Grade', 'Total_Score', 'Final_Score'])
y = df['Grade']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nGrade distribution:")
print(y.value_counts())

### 3.3 Encode Categorical Features

In [ ]:
# Get categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(include=['number']).columns

print(f"Categorical columns: {list(categorical_cols)}")
print(f"Numerical columns: {list(numerical_cols)}")

# One-Hot Encode categorical features
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print(f"\nFeatures shape after encoding: {X.shape}")
print(f"Feature names: {X.columns.tolist()}")

### 3.4 Encode Target Variable

In [ ]:
# Encode target (Grade: A, B, C, D, F -> 0, 1, 2, 3, 4)
le = LabelEncoder()
y = le.fit_transform(y)
num_classes = len(np.unique(y))

print(f"Grade classes: {le.classes_}")
print(f"Number of classes: {num_classes}")
print(f"Encoded target distribution:")
for i, grade in enumerate(le.classes_):
    print(f"  {grade} ({i}): {np.sum(y == i)} samples")

## 4. Train-Test Split and Scaling

In [ ]:
# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features using StandardScaler
# IMPORTANT: Fit only on training data to prevent data leakage
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")

## 5. Build the Neural Network Model

In [ ]:
# Build a Sequential ANN model
model = Sequential([
    # Input layer + First hidden layer (64 neurons, ReLU activation)
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),  # Dropout for regularization (prevents overfitting)
    
    # Second hidden layer (32 neurons, ReLU activation)
    Dense(32, activation='relu'),
    Dropout(0.1),  # Less dropout in deeper layers
    
    # Output layer (num_classes neurons, softmax for multi-class classification)
    Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',  # For integer-encoded labels
    metrics=['accuracy']
)

# Display model architecture
model.summary()

## 6. Train the Model

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,  # Use 20% of training data for validation
    verbose=1
)

print("\nTraining completed!")

## 7. Evaluate the Model

In [ ]:
# Evaluate on test set
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy*100:.2f}%")

## 8. Visualize Training History

In [ ]:
# Plot training history
plt.figure(figsize=(14, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("Training curves saved as 'training_curves.png'")

## 9. Generate Predictions and Confusion Matrix

In [ ]:
# Make predictions on test set
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=le.classes_, 
    yticklabels=le.classes_,
    cbar_kws={'label': 'Count'},
    square=True
)
plt.title('Confusion Matrix - Grade Prediction', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Grade', fontsize=12)
plt.ylabel('Actual Grade', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("Confusion matrix saved as 'confusion_matrix.png'")

## 10. Classification Report

In [ ]:
# Print detailed classification report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Per-class accuracy
print("\n" + "="*60)
print("PER-CLASS ACCURACY")
print("="*60)
for i, grade in enumerate(le.classes_):
    class_mask = y_test == i
    class_accuracy = np.sum((y_test[class_mask] == y_pred[class_mask])) / np.sum(class_mask)
    print(f"Grade {grade}: {class_accuracy*100:.2f}%")

## 11. Visualize Model Architecture (Optional)

In [ ]:
# Try to visualize model architecture
# Note: This requires graphviz to be installed on the system
try:
    from tensorflow.keras.utils import plot_model
    from IPython.display import Image
    
    plot_model(
        model, 
        to_file='model_architecture.png', 
        show_shapes=True, 
        show_layer_names=True,
        rankdir='TB',  # Top to bottom
        dpi=96
    )
    print("Model architecture saved as 'model_architecture.png'")
    display(Image('model_architecture.png'))
    
except Exception as e:
    print(f"Could not save/display model architecture plot: {e}")
    print("\nTo enable this feature, install graphviz:")
    print("  - On Ubuntu/Debian: sudo apt-get install graphviz")
    print("  - On macOS: brew install graphviz")
    print("  - On Windows: Download from https://graphviz.org/download/")
    print("  - Then: pip install pydot graphviz")

## 12. Save the Model (Optional)

In [ ]:
# Save the trained model
model.save('student_grade_prediction_model.keras')
print("Model saved as 'student_grade_prediction_model.keras'")

# Save the scaler and label encoder for future use
import pickle
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print("Scaler saved as 'scaler.pkl'")
print("Label encoder saved as 'label_encoder.pkl'")

## Summary

This notebook demonstrated:
1. Loading and exploring a student performance dataset
2. Preprocessing data (handling missing values, encoding categorical features)
3. Building an Artificial Neural Network with TensorFlow/Keras
4. Training the model with proper train-test split and validation
5. Evaluating performance using accuracy, confusion matrix, and classification report
6. Visualizing training history and results

**Key Improvements Made:**
- Added proper error handling throughout
- Included stratified train-test split for balanced classes
- Fixed the model architecture visualization issue with try-except
- Added detailed comments explaining each step
- Removed redundant pip install cells
- Added proper figure saving with high DPI
- Included model and preprocessing objects saving for deployment

In [ ]:
import os
os.environ["PATH"] += ";E:\\Graphviz\\bin"


In [ ]:
dot - graphviz version ...



In [ ]:
!dot -version


In [ ]:
46